# Text Search Demo — analiza rzeczywistej sesji

Notebook korzysta z rzeczywistego `data.csv` i natywnych word-bboxów zapisanych przez pytracker. Brak gaze nie usuwa trialu z analizy odpowiedzi.

In [ ]:
from pathlib import Path
import ast, json, math, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def find_demo_root():
    start = Path.cwd().resolve()
    candidates = [start, start / "tobii-pytracker-demo", *start.parents]
    for c in candidates:
        if c.name == "tobii-pytracker-demo" and (c / "examples").is_dir():
            return c
        nested = c / "tobii-pytracker-demo"
        if nested.is_dir() and (nested / "examples").is_dir():
            return nested.resolve()
    raise FileNotFoundError("Cannot locate tobii-pytracker-demo from current working directory")

DEMO_ROOT = find_demo_root()
print(f"DEMO_ROOT={DEMO_ROOT}")

def newest_session(root: Path):
    sessions = [p for p in root.iterdir() if p.is_dir() and (p / "data.csv").is_file()] if root.is_dir() else []
    if not sessions:
        raise FileNotFoundError(f"No session with data.csv under {root}")
    return max(sessions, key=lambda p: (p / "data.csv").stat().st_mtime)

def parse_struct(value, expected_type, default):
    if isinstance(value, expected_type):
        return value
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return default
    text = str(value).strip()
    if not text or text.lower() == "nan":
        return default
    for parser in (json.loads, ast.literal_eval):
        try:
            parsed = parser(text)
            if isinstance(parsed, expected_type):
                return parsed
        except Exception:
            pass
    return default

def first_value(d, *names):
    for name in names:
        if name in d and d[name] is not None:
            return d[name]
    return None

def flatten_gaze(raw, set_name):
    records=[]
    for slide_index, row in raw.reset_index(drop=True).iterrows():
        gaze=parse_struct(row.get("gaze_data"), list, [])
        for sample in gaze:
            if not isinstance(sample, dict):
                continue
            x=first_value(sample, "avg_gaze_x", "gaze_x", "x")
            y=first_value(sample, "avg_gaze_y", "gaze_y", "y")
            if x is None:
                lx,rx=sample.get("gaze_x_left"),sample.get("gaze_x_right")
                x=(lx+rx)/2 if lx is not None and rx is not None else (lx if lx is not None else rx)
            if y is None:
                ly,ry=sample.get("gaze_y_left"),sample.get("gaze_y_right")
                y=(ly+ry)/2 if ly is not None and ry is not None else (ly if ly is not None else ry)
            t=first_value(sample, "system_time", "time", "timestamp", "logged_time")
            if x is None or y is None:
                continue
            records.append({"set_name":set_name,"slide_index":int(slide_index),"input_data":row.get("input_data"),
                            "classification":str(row.get("classification","")).lower(),
                            "avg_gaze_x":float(x),"avg_gaze_y":float(y),
                            "system_time":float(t) if t is not None else float(len(records))})
    return pd.DataFrame.from_records(records)

def normalize_token(value):
    return re.sub(r"[^0-9a-ząćęłńóśźż]+", "", str(value).casefold())

def find_target_boxes(objects_bboxes, target_phrase):
    parsed=parse_struct(objects_bboxes,dict,{})
    words=parsed.get("words",[]) if isinstance(parsed,dict) else []
    tokens=[normalize_token(w.get("word","")) for w in words]
    target=[normalize_token(x) for x in str(target_phrase).split()]
    target=[x for x in target if x]
    for start in range(max(0,len(tokens)-len(target)+1)):
        if tokens[start:start+len(target)]==target:
            boxes=[]
            for item in words[start:start+len(target)]:
                b=item.get("bbox",{})
                if {"cx","cy","w","h"} <= set(b): boxes.append({k:float(b[k]) for k in ("cx","cy","w","h")})
            return boxes
    return []

def point_in_boxes(x,y,boxes,padding=8.0):
    return any((b["cx"]-b["w"]/2-padding)<=x<=(b["cx"]+b["w"]/2+padding) and (b["cy"]-b["h"]/2-padding)<=y<=(b["cy"]+b["h"]/2+padding) for b in boxes)


In [ ]:
from tobii_pytracker.analyze import FixationAnalyzer
OUTPUT_ROOT=DEMO_ROOT/"output"/"tobii_text_search_demo"; SESSION=newest_session(OUTPUT_ROOT)
DATASET=DEMO_ROOT/"examples"/"tobii_text_search_demo"/"data"/"text_search.csv"
stimuli=pd.read_csv(DATASET); raw=pd.read_csv(SESSION/"data.csv",sep=";")
if len(raw)!=12: raise RuntimeError(f"Expected 12 trials, got {len(raw)}")
lookup=stimuli.set_index("selected_text",drop=False)
flat=flatten_gaze(raw,SESSION.name); analysis_dir=SESSION/"analysis_text_search"; analysis_dir.mkdir(exist_ok=True); flat.to_csv(analysis_dir/"flattened_gaze.csv",index=False)
fixations=FixationAnalyzer(analysis_dir,method="dispersion").analyze(flat) if not flat.empty else pd.DataFrame()
if not fixations.empty: fixations.to_csv(analysis_dir/"fixations.csv",index=False)
rows=[]
for slide_index,row in raw.reset_index(drop=True).iterrows():
    text=str(row["input_data"]); meta=lookup.loc[text]
    slide_flat=flat[flat["slide_index"]==slide_index] if not flat.empty else pd.DataFrame(); slide_fix=fixations[pd.to_numeric(fixations.get("slide_index"),errors="coerce")==slide_index].copy() if not fixations.empty else pd.DataFrame()
    boxes=find_target_boxes(row.get("objects_bboxes"),meta["critical_phrase"])
    hit=slide_fix.apply(lambda f: point_in_boxes(float(f["x_mean"]),float(f["y_mean"]),boxes),axis=1) if not slide_fix.empty and boxes else pd.Series(False,index=slide_fix.index); target_fix=slide_fix[hit] if not slide_fix.empty else pd.DataFrame()
    trial_start=pd.to_numeric(slide_flat.get("system_time"),errors="coerce").dropna().min() if not slide_flat.empty else np.nan
    if not target_fix.empty:
        first=float(target_fix.iloc[0]["fix_start"]); ttff=first-trial_start if pd.notna(trial_start) else np.nan; dwell=float(pd.to_numeric(target_fix["duration"],errors="coerce").fillna(0).sum()); before=int((pd.to_numeric(slide_fix["fix_start"],errors="coerce")<first).sum())
    else: ttff=np.nan; dwell=0.0; before=int(len(slide_fix))
    expected=str(meta["answer"]).lower(); response=str(row.get("user_classification","")).lower()
    rows.append({"slide_index":slide_index,"item_id":meta["item_id"],"condition":str(meta["condition"]).upper(),"expected":expected,"response":response,"correct":response==expected,"gaze_samples":len(slide_flat),"target_bbox_found":bool(boxes),"target_seen":not target_fix.empty,"ttff_target_s":ttff,"target_dwell_s":dwell,"fixations_before_target":before,"fixation_count":len(slide_fix)})
metrics=pd.DataFrame(rows); metrics.to_csv(analysis_dir/"trial_metrics.csv",index=False)
summary=metrics.groupby("condition",as_index=False).agg(trials=("item_id","size"),accuracy=("correct","mean"),gaze_available_rate=("gaze_samples",lambda s:(s>0).mean()),target_seen_rate=("target_seen","mean"),mean_ttff_target_s=("ttff_target_s","mean"),mean_target_dwell_s=("target_dwell_s","mean"),mean_fixations_before_target=("fixations_before_target","mean"))
summary.to_csv(analysis_dir/"condition_summary.csv",index=False); display(summary)
summary.set_index("condition")[["accuracy"]].plot(kind="bar",ylim=(0,1),title="Text-search accuracy"); plt.show()
print(f"SESSION={SESSION}; gaze_missing_trials={(metrics.gaze_samples==0).sum()}; analysis_dir={analysis_dir}")
print("NATIVE_TEXT_SEARCH_ANALYSIS_PASS")
